In [0]:
CREATE OR REPLACE TABLE default.clean_orders AS
SELECT
  order_id,
  customer_id,
  order_status,
  TO_TIMESTAMP(order_purchase_timestamp) AS order_ts,
  TO_TIMESTAMP(order_delivered_customer_date) AS delivered_ts,
  TO_TIMESTAMP(order_estimated_delivery_date) AS estimated_ts,
  DATEDIFF(
    TO_TIMESTAMP(order_delivered_customer_date),
    TO_TIMESTAMP(order_purchase_timestamp)
  ) AS fulfilment_days
FROM default.olist_orders;


ALTER TABLE default.clean_orders
ADD COLUMN delivery_tier STRING;


UPDATE default.clean_orders
SET delivery_tier =
  CASE
    WHEN fulfilment_days IS NULL THEN 'PENDING'
    WHEN fulfilment_days <= 3 THEN 'FAST'
    WHEN fulfilment_days <= 7 THEN 'MEDIUM'
    ELSE 'SLOW'
  END;


SELECT delivery_tier, COUNT(*) 
FROM default.clean_orders
GROUP BY delivery_tier;